# 第 2 周：NumPy Shape 与 PyTorch 训练循环

**目标**：建立“数据形状 → 矩阵运算 → 张量 → 训练循环”的最短闭环。  
**先修**：完成第 1 周；需要 `numpy` 和 `torch`（Google Colab 默认提供）。

学完后你应该能：

1. 读出数组和张量的 `shape`；
2. 区分逐元素乘法 `*` 与矩阵乘法 `@`；
3. 解释广播发生在哪个维度；
4. 使用 `TensorDataset`、`DataLoader`；
5. 从空白写出 `zero_grad → forward → loss → backward → step`。


In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
print("NumPy", np.__version__, "| PyTorch", torch.__version__)


## 1. Shape 是每个轴的长度

对二维数组 `(rows, columns)`，可把每一行理解为一个样本，每一列理解为一个特征。


In [ ]:
x_np = np.array([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])
bias_np = np.array([10.0, 20.0, 30.0])

print("x shape:", x_np.shape)
print("broadcast result:\n", x_np + bias_np)


In [ ]:
assert x_np.shape == (2, 3)
assert (x_np + bias_np).shape == (2, 3)
assert np.array_equal((x_np + bias_np)[1], [14.0, 25.0, 36.0])
print("shape 与广播测试通过 ✅")


## 2. `*` 与 `@`

- `x * w`：对应位置相乘，通常保持可广播后的形状。
- `x @ w`：行向量与权重列做点积，输入 `(batch, features)` 乘 `(features, outputs)`。

先预测下面输出的 shape，再运行。


In [ ]:
weights_np = np.array([[1.0], [0.5], [-1.0]])
prediction_np = x_np @ weights_np

print("weights:", weights_np.shape)
print("prediction:", prediction_np.shape)
print(prediction_np)


In [ ]:
assert prediction_np.shape == (2, 1)
assert np.allclose(prediction_np[:, 0], [-1.0, 0.5])
print("矩阵乘法测试通过 ✅")


## 3. Dataset 与 DataLoader

`TensorDataset` 把同一行的特征和标签绑在一起；`DataLoader` 决定批次大小及是否打乱。


In [ ]:
x = torch.arange(0, 20, dtype=torch.float32).reshape(-1, 1)
y = 3 * x + 2

dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=5, shuffle=True)

batch_x, batch_y = next(iter(loader))
print(batch_x.shape, batch_y.shape)


In [ ]:
assert len(dataset) == 20
assert batch_x.shape == (5, 1)
assert batch_y.shape == (5, 1)
print("Dataset/DataLoader 测试通过 ✅")


## 4. 最小训练循环

顺序不能乱：

1. `optimizer.zero_grad()` 清除上一步梯度；
2. `prediction = model(batch_x)` 前向计算；
3. `loss = loss_fn(prediction, batch_y)` 衡量误差；
4. `loss.backward()` 计算梯度；
5. `optimizer.step()` 更新参数。


In [ ]:
torch.manual_seed(42)
model = nn.Linear(1, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.005)


def dataset_loss(model, x, y):
    with torch.no_grad():
        return loss_fn(model(x), y).item()


initial_loss = dataset_loss(model, x, y)

for epoch in range(200):
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        prediction = model(batch_x)
        loss = loss_fn(prediction, batch_y)
        loss.backward()
        optimizer.step()

final_loss = dataset_loss(model, x, y)
print(f"loss: {initial_loss:.3f} → {final_loss:.6f}")
print("weight:", model.weight.item(), "bias:", model.bias.item())


In [ ]:
assert final_loss < 0.02
assert abs(model.weight.item() - 3.0) < 0.05
assert abs(model.bias.item() - 2.0) < 0.15
print("训练循环测试通过 ✅")


## 常见错误

- `(2, 3) @ (2, 1)` 不合法：中间维度必须相同。
- 把 batch 维度写丢：模型通常期待 `(batch, features)`。
- 漏掉 `zero_grad()`：梯度会累加。
- 只调用 `backward()` 不调用 `step()`：算出了梯度但没有更新参数。
- 在评估时继续构建梯度图：应使用 `with torch.no_grad()`。


## 扩展挑战与出口测验

- 把输入改为两个特征，构造 `y = 2*x1 - x2 + 1`，同步修改模型维度。
- 闭卷画出一个 batch 经过训练循环的五步。
- 给出 `(32, 10) @ (10, 4)` 的输出 shape，并解释依据。

**通过条件**：全部断言通过，能从空白写训练循环，并正确回答三道 shape 题。
